# Clone Giọng Đọc BLV Tin Tức Bóng Đá — Broadcast Mastering + Từ Điển Phiên Âm Cầu Thủ & SVĐ

1. Bật **Thời gian chạy → Đổi loại thời gian chạy → T4 GPU**.
2. **Đã nhúng sẵn giọng BLV chuẩn sạch (`5.03s`, 24kHz mono)** ngay trong file — không cần upload lại giọng mẫu!
3. **Tối ưu riêng cho Bản Tin Bóng Đá:**
   - **Chống ngắt nghỉ lung tung:** Tự động cắt khoảng lặng thừa ở đầu/cuối mỗi đoạn F5-TTS (`trim_chunk_silence`), chỉ ngắt đúng dấu câu với nhịp nhanh gọn chuẩn bản tin (`.` nghỉ `0.24s`, `,` nghỉ `0.10s`, chuyển đoạn nghỉ `0.42s`).
   - **Tự động phiên âm chuẩn 150+ tên Cầu thủ, HLV, CLB, SVĐ, Giải đấu & Tỷ số:** `Old Trafford`, `Bernabeu`, `Mbappe`, `Haaland`, `Lamine Yamal`, `CAHN`, `CLB`, `HLV`, `SVĐ`, `VAR`, `U23`, `3-1`, `90+4'`, `4-3-3`... (dễ thêm tên mới ngay tại mục `CUSTOM_NAMES`).
   - **Broadcast Mastering:** Giọng sáng, chắc, nén đều chuẩn truyền hình thể thao (`-14 LUFS`).
4. Bấm **▶** → chọn file `.txt` / `.md` tin bóng đá *(tích `PREVIEW_MODE` nếu muốn nghe thử nhanh 3 đoạn đầu)*.


In [ ]:
# @title CHỌN FILE KỊCH BẢN TIN BÓNG ĐÁ (.txt / .md) HOẶC DÁN TRỰC TIẾP ĐỂ XUẤT MP3 GIỌNG BLV
SPEED = 1.0  # @param {type:"number"}
NFE_STEP = 24  # @param [16, 24, 32] {type:"raw"}
PAUSE_SEC = 0.24  # @param {type:"slider", min:0.10, max:0.60, step:0.02}
BROADCAST_MASTERING = True  # @param {type:"boolean"}
PREVIEW_MODE = False  # @param {type:"boolean"}
PREVIEW_CHUNKS = 3  # @param {type:"slider", min:1, max:10, step:1}
SHOW_NORMALIZED_TEXT = True  # @param {type:"boolean"}
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}

# Thêm nhanh tên cầu thủ / CLB / SVĐ mới vào đây nếu chưa có sẵn trong từ điển (không phân biệt hoa thường):
CUSTOM_NAMES = {
    # 'Tên gốc': 'cách đọc tiếng việt',
    'xuân son': 'xuân sơn',
    'quang hải': 'quang hải',
}

import os
import re
import sys
import base64
import shutil
import importlib.util
import subprocess
from pathlib import Path
from importlib.resources import files as pkg_files
from IPython.display import Audio, display
from google.colab import drive, files

# 1. Kết nối Google Drive Cache (tự chuyển sang bộ nhớ tạm Colab nếu Drive không kết nối được)
DRIVE_OK = False
if USE_GOOGLE_DRIVE:
    try:
        if not Path('/content/drive/MyDrive').exists():
            drive.mount('/content/drive')
        DRIVE_OK = Path('/content/drive/MyDrive').exists()
    except Exception as e:
        print(f'⚠️ Không kết nối được Google Drive ({e}). Tự động chạy trực tiếp trên Colab (không cần Drive)!')
        DRIVE_OK = False

if DRIVE_OK:
    CACHE_DIR = Path('/content/drive/MyDrive/Football_BLV_TTS_Cache')
    BIBLE_CACHE = Path('/content/drive/MyDrive/Bible_TTS_Cache')
else:
    CACHE_DIR = Path('/content/Football_BLV_TTS_Cache')
    BIBLE_CACHE = Path('/content/_no_bible_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
PIP_CACHE = CACHE_DIR / 'pip_cache'
PIP_CACHE.mkdir(parents=True, exist_ok=True)
REPO_DIR = Path('/content/F5-TTS-Vietnamese')

# 2. Tải mã nguồn & cài nhanh (~15 giây)
if not REPO_DIR.exists():
    print('>> Đang tải mã nguồn F5-TTS-Vietnamese...')
    subprocess.run(['git', 'clone', '-q', 'https://github.com/nguyenthienhy/F5-TTS-Vietnamese.git', str(REPO_DIR)], check=True)
    pyproject = REPO_DIR / 'pyproject.toml'
    txt = pyproject.read_text(encoding='utf-8')
    for drop in ['"numpy<=1.26.4",', '"gradio>=3.45.2",', '"transformers_stream_generator",', '"bitsandbytes>0.37.0; platform_machine != \'arm64\' and platform_system != \'Darwin\'",']:
        txt = txt.replace(drop, '')
    pyproject.write_text(txt, encoding='utf-8')

os.chdir(REPO_DIR)
if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))

if importlib.util.find_spec('torchdiffeq') is None or importlib.util.find_spec('vocos') is None or importlib.util.find_spec('omegaconf') is None or shutil.which('f5-tts_infer-cli') is None:
    print('>> Đang cài gói F5-TTS (~15-20 giây)...')
    subprocess.run(['pip', 'install', '-q', '--cache-dir', str(PIP_CACHE), '-e', '.', 'omegaconf'], check=True)

import torch
import torchaudio
import numpy as np
import soundfile as sf
from omegaconf import OmegaConf
from tqdm.auto import tqdm as tqdm_cls
from f5_tts.model import DiT, UNetT
from f5_tts.infer.utils_infer import (
    device as f5_device,
    load_vocoder,
    load_model,
    preprocess_ref_audio_text,
    infer_batch_process,
    target_rms,
    cfg_strength,
    sway_sampling_coef,
)

if not torch.cuda.is_available():
    print('⚠️ CẢNH BÁO: Đang chạy CPU! Hãy bật Thời gian chạy -> Đổi loại -> T4 GPU.')
else:
    print(f'✅ GPU sẵn sàng: {torch.cuda.get_device_name(0)}')

# 3. Nạp Model từ Drive Cache (tận dụng luôn cache cũ nếu đã có)
model_dir = REPO_DIR / 'F5-TTS-Vietnamese-ViVoice'
model_dir.mkdir(exist_ok=True)
ckpt_local = model_dir / 'model_last.pt'
vocab_local = model_dir / 'vocab.txt'
ckpt_drive = CACHE_DIR / 'model_last.pt'
vocab_drive = CACHE_DIR / 'vocab.txt'

if not ckpt_local.exists() or not vocab_local.exists():
    if ckpt_drive.exists() and vocab_drive.exists():
        print('>> Đang nạp Model từ Google Drive Cache...')
        shutil.copy(ckpt_drive, ckpt_local)
        shutil.copy(vocab_drive, vocab_local)
    elif (BIBLE_CACHE / 'model_last.pt').exists() and (BIBLE_CACHE / 'vocab.txt').exists():
        print('>> Đang tái sử dụng Model sẵn có từ Bible_TTS_Cache trên Drive...')
        shutil.copy(BIBLE_CACHE / 'model_last.pt', ckpt_local)
        shutil.copy(BIBLE_CACHE / 'vocab.txt', vocab_local)
    else:
        print('>> Đang tải Model từ Hugging Face (chỉ 1 lần đầu)...')
        from huggingface_hub import hf_hub_download
        for fname in ['config.json', 'model_last.pt']:
            hf_hub_download(repo_id='hynt/F5-TTS-Vietnamese-ViVoice', filename=fname, local_dir=str(model_dir))
        shutil.copy(model_dir / 'config.json', vocab_local)
        shutil.copy(ckpt_local, ckpt_drive)
        shutil.copy(vocab_local, vocab_drive)

# 4. Giải nén giọng mẫu BLV Bóng Đá mới đã nhúng sẵn (5.03 giây thuần tiếng Việt, 24kHz mono)
BLV_REF_WAV_B64 = "..."
ref_local = REPO_DIR / 'blv_voice_ref_clean.wav'
ref_local.write_bytes(base64.b64decode(BLV_REF_WAV_B64))
REF_TEXT = 'chào mừng quý vị và các bạn đến với bản tin nóng bóng đá, ngày hai mươi sáu tháng mười một trên kênh bóng đá tổ dân phố.'

# Giữ sẵn Vocoder & Model trên GPU RAM suốt phiên Colab
cache = globals().get('_F5_BLV_CACHE')
if not cache or not Path(cache['ref_audio_proc']).exists() or cache.get('ref_text_raw') != REF_TEXT:
    print('>> Đang đưa Model & Vocoder lên GPU RAM (chỉ 1 lần trong phiên)...')
    vocoder = load_vocoder(vocoder_name='vocos', is_local=False, device=f5_device)
    model_cfg = OmegaConf.load(str(pkg_files('f5_tts').joinpath('configs/F5TTS_Base.yaml'))).model
    model_cls = {'DiT': DiT, 'UNetT': UNetT}[model_cfg.backbone]
    ema_model = load_model(
        model_cls,
        model_cfg.arch,
        str(ckpt_local),
        mel_spec_type='vocos',
        vocab_file=str(vocab_local),
        device=f5_device,
    )
    ref_audio_proc, ref_text_proc = preprocess_ref_audio_text(str(ref_local), REF_TEXT, device=f5_device)
    _F5_BLV_CACHE = {
        'vocoder': vocoder,
        'ema_model': ema_model,
        'ref_audio_proc': ref_audio_proc,
        'ref_text_proc': ref_text_proc,
        'ref_text_raw': REF_TEXT,
    }
    print('✅ Đã giữ sẵn Model & Giọng BLV trên GPU RAM!')
else:
    print('✅ Dùng lại Model & Giọng BLV đang nằm sẵn trên GPU RAM!')

vocoder = _F5_BLV_CACHE['vocoder']
ema_model = _F5_BLV_CACHE['ema_model']
ref_audio_proc = _F5_BLV_CACHE['ref_audio_proc']
ref_text_proc = _F5_BLV_CACHE['ref_text_proc']

# ==============================================================================
# 5. BỘ CHUẨN HÓA PHIÊN ÂM BÓNG ĐÁ (CẦU THỦ, CLB, SVĐ, GIẢI ĐẤU, TỶ SỐ, SƠ ĐỒ)
# ==============================================================================
DIGITS_VI = ['không', 'một', 'hai', 'ba', 'bốn', 'năm', 'sáu', 'bảy', 'tám', 'chín']

def int_to_vi(n: int) -> str:
    if n < 0:
        return 'âm ' + int_to_vi(-n)
    if n < 10:
        return DIGITS_VI[n]
    if n < 20:
        if n == 10: return 'mười'
        if n == 15: return 'mười lăm'
        return 'mười ' + DIGITS_VI[n % 10]
    if n < 100:
        t, u = divmod(n, 10)
        res = DIGITS_VI[t] + ' mươi'
        if u == 1: res += ' mốt'
        elif u == 4: res += ' tư'
        elif u == 5: res += ' lăm'
        elif u > 0: res += ' ' + DIGITS_VI[u]
        return res
    if n < 1000:
        h, r = divmod(n, 100)
        res = DIGITS_VI[h] + ' trăm'
        if r == 0: return res
        if r < 10: return res + ' lẻ ' + DIGITS_VI[r]
        return res + ' ' + int_to_vi(r)
    if n < 10000:
        th, r = divmod(n, 1000)
        res = DIGITS_VI[th] + ' nghìn'
        if r == 0: return res
        if r < 10: return res + ' không trăm lẻ ' + DIGITS_VI[r]
        if r < 100: return res + ' không trăm ' + int_to_vi(r)
        return res + ' ' + int_to_vi(r)
    return ' '.join(DIGITS_VI[int(c)] for c in str(n))

FOOTBALL_PRONUNCIATION_MAP = {
    # --- Viết tắt & Giải đấu ---
    'champions league': 'cham pi ơn lích',
    'europa league': 'ơ rô pa lích',
    'conference league': 'con phơ rần lích',
    'premier league': 'prê mi ơ lích',
    'v-league': 'vê lích',
    'v.league': 'vê lích',
    'la liga': 'la li ga',
    'serie a': 'xê ri a',
    'bundesliga': 'bun đét li ga',
    'ligue 1': 'lích ăng',
    'world cup': 'uôn cúp',
    'asian cup': 'ây dần cúp',
    'aff cup': 'a ép ép cúp',
    'asean cup': 'a xê an cúp',
    'sea games': 'xi gêm',
    'fa cup': 'ép ây cúp',
    'carabao cup': 'ca ra bao cúp',
    'copa america': 'cô pa a mê ri ca',
    'euro': 'ơ rô',
    'fifa': 'phi pha',
    'uefa': 'u ê pha',
    'afc': 'a ép xê',
    'vff': 'vê ép ép',
    'vpf': 'vê pê ép',
    'var': 'va',
    'clb': 'câu lạc bộ',
    'hlv': 'huấn luyện viên',
    'svđ': 'sân vận động',
    'đtqg': 'đội tuyển quốc gia',
    'đtvn': 'đội tuyển việt nam',
    'cđv': 'cổ động viên',
    'bhl': 'ban huấn luyện',
    'tđ': 'tiền đạo',
    'tv': 'tiền vệ',
    'hv': 'hậu vệ',
    'tm': 'thủ môn',
    'vđqg': 'vô địch quốc gia',
    'cahn': 'công an hà nội',
    'hagl': 'hoàng anh gia lai',
    'slna': 'sông lam nghệ an',
    'tphcm': 'thành phố hồ chí minh',
    'tp.hcm': 'thành phố hồ chí minh',

    # --- Sân vận động nổi tiếng ---
    'old trafford': 'ôn tra phót',
    'stamford bridge': 'xtam phót brít',
    'santiago bernabeu': 'xan ti a gô béc na bêu',
    'bernabeu': 'béc na bêu',
    'camp nou': 'cam nâu',
    'anfield': 'an phiu',
    'emirates': 'ê mi rết',
    'etihad': 'ê ti hát',
    'san siro': 'xan xi rô',
    'giuseppe meazza': 'giu dép pê mê át da',
    'allianz arena': 'a li ăn a rê na',
    'signal iduna park': 'xích nồ i đu na pác',
    'parc des princes': 'pác đê prăng xơ',
    'wembley': 'oem bờ lây',
    'metropolitano': 'mê trô pô li ta nô',
    'st james park': 'xanh giêm pác',
    'villa park': 'vi la pác',
    'rajamangala': 'ra gia man ga la',
    'bung karno': 'bung các nô',
    'bukit jalil': 'bu kít gia lin',
    'thammasat': 'tham ma xát',
    'saitama': 'xai ta ma',

    # --- Câu lạc bộ ---
    'manchester united': 'man chét tơ iu nai tít',
    'man utd': 'man chét tơ iu nai tít',
    'man united': 'man chét tơ iu nai tít',
    'manchester city': 'man chét tơ xi ti',
    'man city': 'man chét tơ xi ti',
    'real madrid': 'rê an ma đrít',
    'atletico madrid': 'át lê ti cô ma đrít',
    'barcelona': 'bác xê lô na',
    'barca': 'bác xa',
    'arsenal': 'ác xê nan',
    'liverpool': 'li vơ pun',
    'chelsea': 'chen xi',
    'tottenham hotspur': 'tót ten ham hót xpơ',
    'tottenham': 'tót ten ham',
    'newcastle united': 'niu cát xồ iu nai tít',
    'newcastle': 'niu cát xồ',
    'aston villa': 'át tôn vi la',
    'brighton': 'bờ rai tần',
    'west ham': 'uét ham',
    'crystal palace': 'crít xtan pa lát',
    'nottingham forest': 'nót tinh ham pho rét',
    'everton': 'ê vơ tần',
    'fulham': 'phun ham',
    'brentford': 'bờ ren phót',
    'bournemouth': 'bon mớt',
    'wolves': 'un vơ',
    'leicester': 'lét xtơ',
    'bayern munich': 'bay ơn muy ních',
    'bayern': 'bay ơn',
    'borussia dortmund': 'bô rút xi a đót mun',
    'dortmund': 'đót mun',
    'bayer leverkusen': 'bay ơ lê vơ cu den',
    'leverkusen': 'lê vơ cu den',
    'rb leipzig': 'lai xích',
    'paris saint-germain': 'pa ri xanh giéc manh',
    'paris saint germain': 'pa ri xanh giéc manh',
    'psg': 'pê ét giê',
    'marseille': 'mác xây',
    'lyon': 'li ông',
    'lille': 'lin',
    'monaco': 'mô na cô',
    'inter milan': 'in tơ mi lan',
    'ac milan': 'a xê mi lan',
    'juventus': 'giu ven tút',
    'napoli': 'na pô li',
    'as roma': 'a ét rô ma',
    'atalanta': 'a ta lan ta',
    'lazio': 'la di ô',
    'ajax': 'a dắc',
    'benfica': 'ben phi ca',
    'porto': 'póc tô',
    'sporting lisbon': 'xpót tinh lít bon',
    'al nassr': 'an nát xơ',
    'al hilal': 'an hi lan',
    'inter miami': 'in tơ mai a mi',
    'buriram united': 'bu ri ram iu nai tít',
    'kawasaki frontale': 'ca oa xa ki phờ rôn ta lê',
    'urawa reds': 'u ra oa rét',
    'yokohama': 'dô cô ha ma',
    'vissel kobe': 'vít xeo cô bê',
    'sanfrecce hiroshima': 'xan phờ rét chê hi rô xi ma',
    'mu': 'mờ u',

    # --- HLV & Siêu sao cầu thủ ---
    'pep guardiola': 'pép goác đi ô la',
    'guardiola': 'goác đi ô la',
    'carlo ancelotti': 'các lô an xê lốt ti',
    'ancelotti': 'an xê lốt ti',
    'jose mourinho': 'giô dê mô ri nhô',
    'mourinho': 'mô ri nhô',
    'jurgen klopp': 'dơ ghen cờ lốp',
    'klopp': 'cờ lốp',
    'mikel arteta': 'mi ken ác tê ta',
    'arteta': 'ác tê ta',
    'ruben amorim': 'ru ben a mô rim',
    'amorim': 'a mô rim',
    'arne slot': 'ác nơ xlót',
    'xabi alonso': 'xa bi a lôn xô',
    'hansi flick': 'han xi phờ lích',
    'simone inzaghi': 'xi mô nê in da ghi',
    'diego simeone': 'đi ê gô xi mê ô nê',
    'luis enrique': 'lu ít en ri kê',
    'kim sang-sik': 'kim xang xích',
    'kim sang sik': 'kim xang xích',
    'park hang-seo': 'pác hang xo',
    'park hang seo': 'pác hang xo',
    'mano polking': 'ma nô pôn kinh',
    'cristiano roland': 'crít xti a nô rô lăng',
    'polking': 'pôn kinh',
    'cristiano ronaldo': 'crít xti a nô rô nan đô',
    'ronaldo': 'rô nan đô',
    'lionel messi': 'li ô nen mét xi',
    'messi': 'mét xi',
    'kylian mbappe': 'ki li an em ba pê',
    'mbappe': 'em ba pê',
    'mbappé': 'em ba pê',
    'erling haaland': 'ơ linh ha lân',
    'haaland': 'ha lân',
    'jude bellingham': 'giút be linh ham',
    'bellingham': 'be linh ham',
    'vinicius junior': 'vi ni xi út giu ni ơ',
    'vinicius': 'vi ni xi út',
    'lamine yamal': 'la min da man',
    'yamal': 'da man',
    'mohamed salah': 'mô ha mét xa la',
    'salah': 'xa la',
    'kevin de bruyne': 'kê vin đờ bờ rơ nơ',
    'de bruyne': 'đờ bờ rơ nơ',
    'bukayo saka': 'bu ca dô xa ca',
    'saka': 'xa ca',
    'cole palmer': 'côn pam mơ',
    'palmer': 'pam mơ',
    'phil foden': 'phiu phô đần',
    'foden': 'phô đần',
    'rodri': 'rô đri',
    'bruno fernandes': 'bờ ru nô phéc nan đét',
    'fernandes': 'phéc nan đét',
    'marcus rashford': 'mác cút rát phót',
    'rashford': 'rát phót',
    'harry kane': 'ha ri kên',
    'kane': 'kên',
    'robert lewandowski': 'rô béc lê oan đốp xki',
    'lewandowski': 'lê oan đốp xki',
    'pedri': 'pê đri',
    'raphinha': 'ra phi nha',
    'modric': 'mô đrích',
    'valverde': 'van véc đê',
    'rodrygo': 'rô đri gô',
    'courtois': 'cuốc toa',
    'van dijk': 'van đai',
    'alisson': 'a li xơn',
    'son heung-min': 'xon hưng min',
    'son heung min': 'xon hưng min',
    'kaoru mitoma': 'ca ô ru mi tô ma',
    'mitoma': 'mi tô ma',
    'takefusa kubo': 'ta kê phu xa cu bô',
    'kubo': 'cu bô',
    'neymar': 'nây ma',
    'griezmann': 'gri ét man',
    'lautaro martinez': 'lau ta rô mác ti nét',
    'florian wirtz': 'phờ lo ri an uốt',
    'musiala': 'mu xi a la',
    'declan rice': 'đếch lân rai',
    'odegaard': 'ô đờ gát',
    'saliba': 'xa li ba',
    'gyokeres': 'di ô kê rét',
    'isak': 'i xác',
    'rafael son': 'ra pha en xơn',
}

def normalize_football_text(raw: str) -> str:
    kept = []
    for ln in raw.splitlines():
        s = ln.strip()
        if not s or s.startswith('#') or set(s) <= {'-', '_', '='}:
            continue
        s = s.replace('**', '').replace('*', '').replace('`', '')
        kept.append(s)
    text = ' ||PARA|| '.join(kept)

    # Giữ tag [pause] nếu người dùng chủ động chèn
    text = re.sub(r'\[pause\]', ' ||PAUSE|| ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[[^\]]+\]\s*', '', text)

    # 1. Phút bù giờ: 90+4' hoặc 45+2
    text = re.sub(
        r'\b(\d{1,3})\s*\+\s*(\d{1,2})\s*[\'’]?',
        lambda m: f"{int_to_vi(int(m.group(1)))} cộng {int_to_vi(int(m.group(2)))}",
        text
    )
    # 2. Phút thi đấu: phút 85' hoặc 85'
    text = re.sub(
        r'\b(\d{1,3})\s*[\'’]',
        lambda m: f"phút thứ {int_to_vi(int(m.group(1)))}",
        text
    )
    # 3. Sơ đồ chiến thuật 3 hoặc 4 số: 4-3-3, 4-2-3-1
    text = re.sub(
        r'\b(\d)-(\d)-(\d)(?:-(\d))?\b',
        lambda m: ' '.join(DIGITS_VI[int(g)] for g in m.groups() if g is not None),
        text
    )
    # 4. Khoảng cách 16m50, 11m, 5m50
    text = re.sub(r'\b16m50\b', 'mười sáu mét năm mươi', text, flags=re.IGNORECASE)
    text = re.sub(r'\b5m50\b', 'năm mét năm mươi', text, flags=re.IGNORECASE)
    text = re.sub(r'\b(\d{1,3})m\b', lambda m: f"{int_to_vi(int(m.group(1)))} mét", text, flags=re.IGNORECASE)
    # 5. Đội trẻ U23, U19, U20, U17
    text = re.sub(r'\bU[- ]?(\d{2})\b', lambda m: f"u {int_to_vi(int(m.group(1)))}", text, flags=re.IGNORECASE)
    # 6. Vòng 1/8, 1/4, 1/2
    text = re.sub(r'\bvòng\s*1/8\b', 'vòng một phần tám', text, flags=re.IGNORECASE)
    text = re.sub(r'\bvòng\s*1/4\b', 'vòng tứ kết', text, flags=re.IGNORECASE)
    text = re.sub(r'\b1/8\b', 'một phần tám', text)
    # 7. Tỷ số trận đấu: 3-1, 2–0, 1-1
    def _score_repl(m):
        a, b = int(m.group(1)), int(m.group(2))
        if a <= 15 and b <= 15:
            return f"{int_to_vi(a)} {int_to_vi(b)}"
        return f"{int_to_vi(a)} đến {int_to_vi(b)}"
    text = re.sub(r'\b(\d{1,2})\s*[-–—:]\s*(\d{1,2})\b', _score_repl, text)
    # 8. Phần trăm: 75%
    text = re.sub(r'\b(\d{1,3})\s*%', lambda m: f"{int_to_vi(int(m.group(1)))} phần trăm", text)

    # 9. Thay thế từ điển bóng đá + CUSTOM_NAMES (ưu tiên cụm dài trước, khớp nguyên từ)
    merged_map = {**FOOTBALL_PRONUNCIATION_MAP, **{k.lower(): v.lower() for k, v in CUSTOM_NAMES.items()}}
    text = text.replace('—', ', ').replace('–', ', ').replace(':', ', ').replace(';', ', ')
    text = text.replace('"', '').replace('“', '').replace('”', '').replace('(', ', ').replace(')', ', ')
    text = text.lower()

    for k in sorted(merged_map.keys(), key=len, reverse=True):
        pat = r'(?<![a-zà-ỹđ0-9])' + re.escape(k) + r'(?![a-zà-ỹđ0-9])'
        text = re.sub(pat, merged_map[k], text)

    # 10. Chuyển các con số còn lại sang chữ tiếng Việt
    text = re.sub(r'\b(\d{1,4})\b', lambda m: int_to_vi(int(m.group(1))), text)
    # Bỏ gạch nối giữa các chữ để F5-TTS đọc liền mạch
    text = re.sub(r'(?<=[a-zà-ỹđ])-(?=[a-zà-ỹđ])', ' ', text)
    text = re.sub(r'\s+,', ',', text)
    text = re.sub(r',\s*,+', ', ', text)
    return re.sub(r'\s+', ' ', text).strip()

def smart_chunk_football(text: str, target_chars: int = 145, max_chars: int = 195):
    """Cắt câu chuẩn nhịp bản tin bóng đá: không cắt giữa vế, không tạo mảnh vụn gây ngắt nghỉ lung tung."""
    # Chuẩn hóa mốc ngắt đoạn
    raw_blocks = re.split(r'\|\|(?:pause|para)\|\|', text, flags=re.IGNORECASE)
    blocks = [b.strip() for b in raw_blocks if b.strip()]
    chunks = []
    for b_idx, block in enumerate(blocks):
        sents = [s.strip() for s in re.split(r'(?<=[.?!])\s+', block) if s.strip()]
        norm_sents = []
        for s in sents:
            s_clean = re.sub(r'\s+', ' ', s).strip()
            if not s_clean:
                continue
            while len(s_clean) > max_chars and ',' in s_clean:
                mid = len(s_clean) // 2
                commas = [m.start() for m in re.finditer(r',', s_clean)]
                split_at = min(commas, key=lambda pos: abs(pos - mid))
                left = s_clean[:split_at + 1].strip()
                right = s_clean[split_at + 1:].strip()
                if len(left) < 35 or len(right) < 35:
                    break
                norm_sents.append(left)
                s_clean = right
            norm_sents.append(s_clean)

        cur = ''
        for s_txt in norm_sents:
            if not cur:
                cur = s_txt
            elif (len(cur) + 1 + len(s_txt) <= target_chars) and not cur.endswith(('?', '!')):
                cur = cur + ' ' + s_txt
            else:
                chunks.append((cur, False))
                cur = s_txt
        if cur:
            chunks.append((cur, b_idx < len(blocks) - 1))

    result = []
    base_p = float(PAUSE_SEC)
    for seg_txt, is_para_end in chunks:
        if is_para_end:
            p_sec = max(0.42, base_p * 1.8)
        elif seg_txt.endswith(('?', '!')):
            p_sec = base_p + 0.12
        elif seg_txt.endswith(','):
            p_sec = min(0.10, base_p * 0.45)
        else:
            p_sec = base_p
        result.append((seg_txt, round(p_sec, 2)))
    return result

def trim_chunk_silence(wave: np.ndarray, sr: int, thresh_db: float = -42.0, pad_ms: int = 20) -> np.ndarray:
    """Cắt bỏ khoảng lặng thừa ở đầu/cuối mỗi đoạn F5-TTS để chống hiện tượng ngắt nghỉ lung tung."""
    if len(wave) == 0:
        return wave
    thresh = 10.0 ** (thresh_db / 20.0)
    idx = np.where(np.abs(wave) > thresh)[0]
    if len(idx) == 0:
        return wave
    pad = int(sr * pad_ms / 1000)
    start = max(0, int(idx[0]) - pad)
    end = min(len(wave), int(idx[-1]) + pad)
    return wave[start:end]

# 6. Upload file kịch bản tin bóng đá (.txt / .md)
print('>> Bấm Choose Files bên dưới để chọn file kịch bản tin bóng đá (.txt hoặc .md)...')
uploaded_scripts = files.upload()

ref_tensor, ref_sr = torchaudio.load(ref_audio_proc)
ref_sec = ref_tensor.shape[-1] / ref_sr
max_chars = max(90, int(len(ref_text_proc.encode('utf-8')) / ref_sec * (20 - ref_sec)))

for fname in uploaded_scripts.keys():
    src_path = Path(fname)
    raw_txt = src_path.read_text(encoding='utf-8')
    src_path.unlink(missing_ok=True)
    clean_txt = normalize_football_text(raw_txt)
    if not clean_txt:
        print(f'⚠️ Bỏ qua {fname} vì không có nội dung.')
        continue

    if SHOW_NORMALIZED_TEXT:
        preview_norm = clean_txt.replace('||para||', '\n').replace('||pause||', ' [NGHỈ] ')
        print('\n--- VĂN BẢN ĐÃ CHUẨN HÓA PHIÊN ÂM BÓNG ĐÁ (600 ký tự đầu) ---')
        print(preview_norm[:600] + ('...' if len(preview_norm) > 600 else ''))
        print('--------------------------------------------------------------')

    stem = Path(fname).stem.replace('_script', '')
    chunk_items = smart_chunk_football(clean_txt, target_chars=min(145, max_chars), max_chars=min(195, max_chars + 25))
    total_batches = len(chunk_items)

    if PREVIEW_MODE:
        chunk_items = chunk_items[:int(PREVIEW_CHUNKS)]
        out_mp3 = f'{stem}_blv_preview.mp3'
        print(f'\n>> [PREVIEW MODE] Đang đọc {len(chunk_items)}/{total_batches} đoạn đầu của {fname} | SPEED={SPEED} | PAUSE={PAUSE_SEC}s')
    else:
        out_mp3 = f'{stem}_blv.mp3'
        wc = len(re.sub(r'\|\|(?:para|pause)\|\|', '', clean_txt, flags=re.I).split())
        est_min = wc / (215.0 * float(SPEED))
        print(f'\n>> Đang đọc FULL BLV {fname}: {wc} từ | {total_batches} đoạn | SPEED={SPEED} | PAUSE={PAUSE_SEC}s (~{est_min:.1f} phút)')

    wave_parts = []
    final_sr = 24000
    for seg_txt, pause_after in tqdm_cls(chunk_items, unit='đoạn'):
        seg_wave, final_sr, _ = next(
            infer_batch_process(
                (ref_tensor, ref_sr),
                ref_text_proc,
                [seg_txt],
                ema_model,
                vocoder,
                mel_spec_type='vocos',
                progress=None,
                target_rms=target_rms,
                cross_fade_duration=0.0,
                nfe_step=int(NFE_STEP),
                cfg_strength=cfg_strength,
                sway_sampling_coef=sway_sampling_coef,
                speed=float(SPEED),
                fix_duration=None,
                device=f5_device,
            )
        )
        seg_wave = trim_chunk_silence(np.asarray(seg_wave, dtype=np.float32), final_sr)
        fade_n = min(int(final_sr * 0.012), len(seg_wave) // 4)
        if fade_n > 0:
            seg_wave[:fade_n] *= np.linspace(0.0, 1.0, fade_n, dtype=np.float32)
            seg_wave[-fade_n:] *= np.linspace(1.0, 0.0, fade_n, dtype=np.float32)
        wave_parts.append(seg_wave)
        wave_parts.append(np.zeros(int(final_sr * pause_after), dtype=np.float32))

    final_wave = np.concatenate(wave_parts) if wave_parts else np.zeros(1, dtype=np.float32)
    out_audio_dir = REPO_DIR / 'out_audio_blv'
    shutil.rmtree(out_audio_dir, ignore_errors=True)
    out_audio_dir.mkdir(exist_ok=True)
    raw_wav = out_audio_dir / f'{stem}.wav'
    sf.write(str(raw_wav), final_wave, final_sr)

    # Broadcast BLV Mastering: sáng rõ, chắc giọng, chuẩn âm lượng YouTube -14 LUFS
    af_chain = (
        'highpass=f=80,'
        'equalizer=f=160:t=q:w=1.2:g=2.0,'
        'equalizer=f=3200:t=q:w=1.4:g=2.2,'
        'equalizer=f=6800:t=q:w=2.0:g=-2.5,'
        'acompressor=threshold=-18dB:ratio=3.0:attack=10:release=140:makeup=2.5,'
        'loudnorm=I=-14:TP=-1.0:LRA=8'
    ) if BROADCAST_MASTERING else 'loudnorm=I=-14:TP=-1.0:LRA=10'

    subprocess.run([
        'ffmpeg', '-y', '-loglevel', 'warning', '-i', str(raw_wav),
        '-af', af_chain,
        '-ar', '44100', '-ac', '2', '-b:a', '192k', out_mp3
    ], check=True)

    if DRIVE_OK:
        shutil.copy(out_mp3, CACHE_DIR / out_mp3)
    if PREVIEW_MODE:
        print(f'🎧 Nghe thử {out_mp3} ngay bên dưới (bỏ tích PREVIEW_MODE để xuất bản full):')
        display(Audio(out_mp3))
    else:
        saved_note = 'Đã lưu Drive và đang tải về máy...' if DRIVE_OK else 'Đang tải trực tiếp về máy...'
        print(f'✅ Hoàn tất {out_mp3}! {saved_note}')
        files.download(out_mp3)
